In [ ]:
%load_ext autoreload
%autoreload 2
%config Completer.use_jedi = False

In [ ]:
import numpy as np
import os, glob, pickle, yaml

from matplotlib import pyplot as plt
from pylab import *

In [ ]:
import sys
sys.path.append('./code')

In [ ]:
import torch 
from torch.utils.data import DataLoader
from models import XASLightningModule
from data import XASGraphDataset, collate_fn

In [ ]:
from pymatgen.io.ase import AseAtomsAdaptor
from collections import Counter

In [ ]:
import umap

In [ ]:
ROOT_PATH = './'

In [ ]:
nblocks, cutoff, threebody_cutoff = 3, 4.0, 4.0

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
with open(ROOT_PATH + "/configs/config.yaml", "r") as f:
    config = yaml.safe_load(f)

In [ ]:
GRID=np.linspace(9500-600, 9650-600, 1000)
energy_grid = GRID[333:733:2]

In [ ]:
r1, r2, rZn = 3.0, 5.0, 4.2

### 1. Load data

In [ ]:
structures = pickle.load(open(ROOT_PATH + '/dataset/all_structures.pkl', 'rb'))

In [ ]:
spectra = torch.load(ROOT_PATH + "/dataset/all_spectra.pt")

### 2. Get neighbor counts

In [ ]:
def get_neighbors(structure, r1=3.0, r2=5.0, rZn=4.2):
    atoms = AseAtomsAdaptor.get_atoms(structure)
    zn_idx = 0

    # Get first neighbors (nearest neighbors)
    first_neighbors = structure.get_neighbors(structure[zn_idx], r=r1)  # Using 3.0 Angstrom cutoff
    first_neighbors = [neighbor for neighbor in first_neighbors if neighbor[0].specie.symbol != 'H']
    first_neighbor_species = [neighbor[0].specie.symbol for neighbor in first_neighbors]
    # Get second neighbors
    two_shells_neighbors = structure.get_neighbors(structure[zn_idx], r=r2)  # Using 5.0 Angstrom cutoff
    second_neighbors = [neighbor for neighbor in two_shells_neighbors if neighbor[0].specie.symbol != 'H' and neighbor[0] not in first_neighbors]
    second_neighbor_species = [neighbor[0].specie.symbol for neighbor in second_neighbors]
    
    Zn_shell_neighbors = structure.get_neighbors(structure[zn_idx], r=rZn)  # Using 5.0 Angstrom cutoff
    Zn_neighbors = [neighbor for neighbor in Zn_shell_neighbors if neighbor[0].specie.symbol == 'Zn']
    Zn_neighbors_species = [neighbor[0].specie.symbol for neighbor in Zn_neighbors]
    
    c1 = Counter(first_neighbor_species)
    c2 = Counter(second_neighbor_species)
    cZn = Counter(Zn_neighbors_species)
    arr = []
    for key in ['Cl', 'O']:
        arr.append(c1[key])
    for key in ['Cl', 'O']:
        arr.append(c2[key])
    for key in ['Zn']:
        arr.append(cZn[key])
    return arr # return the coordination number of the structure with shape (5,), 5 are the coordination number of the Cl, O in the first shell, and Cl, O, Zn in the second shell.

In [ ]:
cn_Cl = np.zeros(len(structures)) # list of coordination number of Cl in first shell
cn_O = np.zeros(len(structures)) # list of coordination number of O in first shell
cn_Cl_2nd = np.zeros(len(structures))
cn_O_2nd = np.zeros(len(structures))
cn_Zn = np.zeros(len(structures))

for idx, structure in enumerate(structures): 
        neighbors = get_neighbors(structure)
        cn_Cl[idx]=neighbors[0]
        cn_O[idx]=neighbors[1]
        cn_Cl_2nd[idx]=neighbors[2]
        cn_O_2nd[idx]=neighbors[3]
        cn_Zn[idx]=neighbors[4]

### 3. Convert structure into latent features

In [ ]:
def cache_features(gnn, dataloader, save_path="absorber_features.pt", device="cuda"):
    gnn.eval().to(device)
    feats_all, spectra_all = [], []

    with torch.no_grad():
        for g, _, spectra in dataloader:
            g = g.to(device)
            spectra = spectra.to(device)
            feats = gnn(g)  # (B, d)
            feats_all.append(feats.cpu())
            spectra_all.append(spectra.cpu())

    feats_all = torch.cat(feats_all, dim=0)
    spectra_all = torch.cat(spectra_all, dim=0)
    torch.save((feats_all, spectra_all), save_path)
    print(f"Saved cached features to {save_path}")

In [ ]:
dataset = XASGraphDataset(structures, spectra, cutoff=cutoff)
data_loader = DataLoader(dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

In [ ]:
gnn_config = dict(
    nblocks = config['gnn']['nblocks'], 
    cutoff = config['gnn']['cutoff'], 
    threebody_cutoff = config['gnn']['threebody_cutoff']
)

head_config = dict(
    hidden_dims = [64, 64], 
    output_size = config['head']['output_size'], 
    drop_rate = config['head']['drop_rate'], 
)

model = XASLightningModule(gnn_config, head_config, learning_rate=config['training']['lr'])

In [ ]:
model.load_state_dict(torch.load(os.path.join('./model', "gnn.pth"), weights_only=True))

In [ ]:
# features_file = "absorber_features.pt"
# cache_features(model.gnn, data_loader, save_path="./all_" + features_file, device=device)

### 4. UMAP

In [ ]:
features_file = "absorber_features.pt"
all_feats, all_spectra = torch.load("./data/all_" + features_file)

In [ ]:
spectra = all_spectra.numpy()

In [ ]:
umap_model = umap.UMAP(n_components=2, random_state=1)

In [ ]:
umap_all = umap_model.fit_transform(all_feats)  # Reduce to 2D

In [ ]:
# np.save("./data/umap.npy", umap_all)

#### 4.1 UMAP - plot 

In [ ]:
scatter = plt.scatter(umap_all[:, 0], umap_all[:, 1], s=2, c = cn_Cl, vmax=4.5)#, alpha=0.8)
plt.colorbar(label='num of Cl in FSS')
plt.xlabel('UMAP Dimension 1')
plt.ylabel('UMAP Dimension 2')
# plt.savefig('./figures/UMAP-%s-%s-%s-Cl.png'%(nblocks, cutoff, threebody_cutoff), bbox_inches='tight', dpi=100)

In [ ]:
scatter = plt.scatter(umap_all[:, 0], umap_all[:, 1], s=2, c = cn_O, cmap='plasma', vmax=6.5)#, alpha=0.8)
plt.colorbar(label='num of O in FSS')
plt.xlabel('UMAP Dimension 1')
plt.ylabel('UMAP Dimension 2')
# plt.savefig('./figures/UMAP-%s-%s-%s-O.png'%(nblocks, cutoff, threebody_cutoff), bbox_inches='tight', dpi=100)

In [ ]:
scatter = plt.scatter(umap_all[:, 0], umap_all[:, 1], s=2, c = cn_Cl+ cn_O, cmap='rainbow')#, alpha=0.8)
plt.colorbar(label='Coordination number')
plt.xlabel('UMAP Dimension 1')
plt.ylabel('UMAP Dimension 2')
# plt.savefig('./figures/UMAP-%s-%s-%s-CN.png'%(nblocks, cutoff, threebody_cutoff), bbox_inches='tight', dpi=100)

### 5. UMAP - cluster

In [ ]:
from sklearn.cluster import KMeans
import pandas as pd

In [ ]:
X = umap_all  # from UMAP

In [ ]:
k = 6
kmeans = KMeans(n_clusters=k, random_state=2).fit(X)
original_cluster_labels = kmeans.labels_
original_centers = kmeans.cluster_centers_

In [ ]:
df = pd.DataFrame({
    "cluster": original_cluster_labels,
    "nCl": cn_Cl,
    "nO": cn_O,
    "cn_total": cn_O+cn_Cl,
})

summary = df.groupby("cluster").agg(["mean", "std", "count"])
print(summary)

In [ ]:
order = df.groupby("cluster")["nO"].mean().sort_values().index.tolist()[::-1]

In [ ]:
# Build mapping from old label → rank order
mapping = {old: new for new, old in enumerate(order)}
cluster_labels = np.array([mapping[i] for i in original_cluster_labels])
centers = original_centers[order]

In [ ]:
# np.savetxt("./data/CN-FSS-SSS.txt", 
#            np.column_stack((cn_Cl, cn_O, cn_Cl_2nd, cn_O_2nd, cn_Zn, cluster_labels)), 
#           header = "cn_Cl, cn_O, cn_Cl_2nd, cn_O_2nd, cn_Zn, cluster_labels")

#### Plot cluster

In [ ]:
plt.figure(figsize=(6,5))
plt.scatter(X[:,0], X[:,1], c=cluster_labels, cmap='tab10', s=8, vmin=-0.5, vmax=9.5)
plt.colorbar(ticks=np.arange(6))
plt.scatter(centers[:,0], centers[:,1], c='black', s=50, marker='x')
plt.title(f"UMAP Clusters (k={k})")
plt.xlabel("UMAP-1"); plt.ylabel("UMAP-2")
# plt.savefig('./figures/UMAP-%s-%s-%s-clusters.png'%(nblocks, cutoff, threebody_cutoff), bbox_inches='tight', dpi=100)

In [ ]:
nE = spectra.shape[1]
cluster_means = []
cluster_reps = []

for i in range(k):
    idx = np.where(cluster_labels == i)[0]
    mean_spec = spectra[idx].mean(axis=0)
    cluster_means.append(mean_spec)

    # representative 
    dists = np.sum(mean_spec*spectra[idx], axis=1) / np.linalg.norm(mean_spec) / np.linalg.norm(spectra[idx], axis=1)
    rep_idx = idx[np.argmax(dists)]
    cluster_reps.append(rep_idx)

In [ ]:
plt.figure(figsize=(6,4))
for i, mean_spec in enumerate(cluster_means):
    plt.plot(energy_grid, mean_spec, label=f"Cluster {i}")
# plt.xlabel("Energy (eV)")
plt.xlabel("Energy idx")
plt.ylabel("Intensity")
plt.legend()
plt.title("Mean DFT spectra in each cluster")
plt.xlim(energy_grid[0],energy_grid[-1])
# plt.savefig('./figures/UMAP-mean-sp.png', bbox_inches='tight', dpi=100)

In [ ]:
plt.figure(figsize=(6,4))
for i, rep_idx in enumerate(cluster_reps):
    plt.plot(energy_grid, spectra[rep_idx], label=f"Cluster {i}")
plt.xlabel("Energy idx")
plt.ylabel("Intensity")
plt.legend()
plt.title("Representative DFT spectra in each cluster")
plt.xlim(energy_grid[0],energy_grid[-1])
# plt.savefig('./figures/UMAP-rep-sp.png', bbox_inches='tight', dpi=100)

In [ ]:
# for i, mean_spec in enumerate(cluster_means):
#     np.savetxt("./data/mean_spec-cluster_%s.txt"%i, np.column_stack((energy_grid, mean_spec)) )
# for i, rep_idx in enumerate(cluster_reps):
#     np.savetxt("./data/rep_spec-cluster_%s.txt"%i, np.column_stack((energy_grid, spectra[rep_idx])) )

In [ ]:
df = pd.DataFrame({
    "cluster": cluster_labels,
    "nCl": cn_Cl,
    "nO": cn_O,
    "cn_total": cn_O+cn_Cl,
})

summary = df.groupby("cluster").agg(["mean", "std", "count"])
print(summary)

In [ ]:
cluster_summary = {
    'nCl-mean': summary['nCl']['mean'], 
    'nCl-std': summary['nCl']['std'], 
    'nO-mean': summary['nO']['mean'], 
    'nO-std': summary['nO']['std'], 
    'CN-mean': summary['cn_total']['mean'], 
    'CN-mean': summary['cn_total']['std'], 
    'rep-nCl': [cn_Cl[rep_idx] for rep_idx in cluster_reps], 
    'rep-nO': [cn_O[rep_idx] for rep_idx in cluster_reps], 
}

In [ ]:
# pickle.dump(cluster_summary, open('./data/cluster_summary.pkl', 'wb'), protocol=4)

In [ ]:
for i, rep_idx in enumerate(cluster_reps):
    structure = structures[rep_idx]
    print(i, rep_idx, cn_Cl[rep_idx], cn_O[rep_idx])


In [ ]:
cluster_summary